# B3 · Attention et Transformers : un mini-GPT from scratch — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b3_transformers/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B3.Q1 à B3.Q12, B3.R1 à B3.R3, B3.1 à B3.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B3.12 à B3.17 | ta librairie NumPy : masques et attention, carte d'attention à prédire, têtes, positions sinusoïdales et RoPE, bloc Transformer complet validé contre PyTorch, trois masques qui mentent (*prochaine session de génération*) | | | |
| B | B3.18 à B3.30 | le mini-GPT en PyTorch : module d'attention, modèle, entraînement sur Holmes, courbes, Holmes amorcé en français, KV cache, cartes d'attention, ablations, BPE contre caractères, contexte long, ViT, test de causalité, défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Calculer à la main une attention par produit scalaire, avec et sans masque causal, et justifier la division par $\sqrt{d_k}$.
- Implémenter en NumPy les masques, l'attention multi-têtes, les encodages de position (sinusoïdal, RoPE) et un bloc Transformer pré-LN, et les valider contre PyTorch.
- Écrire et entraîner en PyTorch un mini-GPT sur Holmes, le faire générer avec un KV cache, et comparer sa perplexité à celle du LSTM du ch. 22.
- Estimer les paramètres, les FLOPs et la mémoire d'un Transformer, et mesurer le coût d'un long contexte.

**Rappel express.** $\operatorname{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \operatorname{softmax}\big(\mathbf{Q}\mathbf{K}^\top / \sqrt{d_k} + \mathbf{M}\big)\mathbf{V}$, la softmax sur les clés, $M_{ij} = -\infty$ là où le masque vaut `False`. Masque causal : la requête $i$ voit les clés $j \le i + S - L$. Multi-têtes : $(N, T, d) \to (N, H, T, d_h)$, $4d^2 + 4d$ paramètres. Encodage sinusoïdal : $PE_{p, 2i} = \sin(p\,\omega_i)$, $PE_{p, 2i+1} = \cos(p\,\omega_i)$, $\omega_i = 10000^{-2i/d}$. RoPE : $\langle R_m \mathbf{q}, R_n \mathbf{k} \rangle = \langle \mathbf{q}, R_{n-m} \mathbf{k} \rangle$. Bloc pré-LN : $\mathbf{h} = \mathbf{x} + \operatorname{MHA}(\operatorname{LN}_1(\mathbf{x}))$, $\mathbf{y} = \mathbf{h} + \operatorname{MLP}(\operatorname{LN}_2(\mathbf{h}))$, environ $12d^2$ paramètres. Entraînement : environ $6ND$ FLOPs ; KV cache : $2 \times L \times T \times d \times$ octets par valeur.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B3.1 à B3.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une forme de tableau : entre guillemets (`"(2, 3)"`) ; un vecteur ou une ligne de matrice : une liste (`[0.5, -1, 3]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ B3.6 et B3.7, l'estimation 🧮 B3.8, la réflexion (🗣️ B3.9, ⚖️ B3.10, 📄 B3.11) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np


def softmax_rows(s):
    """Stable softmax over the last axis (the keys); -inf gives a weight of exactly 0."""
    s = np.asarray(s, dtype=float)
    e = np.exp(s - s.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


# ✏️ B3.1: three tokens, d_k = 2 (token i is row i), no mask
X_B31 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
WQ_B31 = np.array([[0.0, 1.0], [1.0, 1.0]])
WK_B31 = np.array([[1.0, 0.0], [1.0, 2.0]])
WV_B31 = np.array([[1.0, -1.0], [0.0, 2.0]])
Q_B31, K_B31, V_B31 = X_B31 @ WQ_B31, X_B31 @ WK_B31, X_B31 @ WV_B31
RAW_B31 = Q_B31 @ K_B31.T                                           # (L, S) dot products
S_B31 = RAW_B31 / math.sqrt(2)                                      # scaled scores
A_B31 = softmax_rows(S_B31)                                         # attention map, softmax over the keys
O_B31 = A_B31 @ V_B31
A_NOSCALE_B31 = softmax_rows(RAW_B31)
A_COLUMNS_B31 = softmax_rows(S_B31.T).T                             # the classic mistake: softmax over the queries

# ✏️ B3.2: the same with a causal mask, then L = 2 new queries against S = 3 keys (KV cache)
CAUSAL_B32 = np.tril(np.ones((3, 3), dtype=bool))
TRIU_B32 = np.triu(np.ones((3, 3), dtype=bool))                     # the transposed mask of 🐛 B3.17
A_C_B32 = softmax_rows(np.where(CAUSAL_B32, S_B31, -np.inf))
O_C_B32 = A_C_B32 @ V_B31
A_ZERO_B32 = np.where(CAUSAL_B32, A_B31, 0.0)                       # the classic mistake: zeroed, not renormalized
A_NOSCALE_C_B32 = softmax_rows(np.where(CAUSAL_B32, RAW_B31, -np.inf))
A_TRIU_B32 = softmax_rows(np.where(TRIU_B32, S_B31, -np.inf))
BOTTOM_RIGHT_B32 = np.tril(np.ones((2, 3), dtype=bool), k=1)        # causal_mask(2, 3)
TOP_LEFT_B32 = np.tril(np.ones((2, 3), dtype=bool), k=0)            # is_causal=True of SDPA when L != S

# ✏️ B3.3: shapes and parameters of a multi-head attention
N_B33, T_B33, D_B33, H_B33, S_B33, HKV_B33 = 4, 10, 384, 6, 25, 2
DH_B33 = D_B33 // H_B33
MHA_B33 = 4 * D_B33 ** 2 + 4 * D_B33
GQA_B33 = 2 * (D_B33 * D_B33 + D_B33) + 2 * (D_B33 * HKV_B33 * DH_B33 + HKV_B33 * DH_B33)

# ✏️ B3.4: sinusoidal encoding, d = 8 (angular frequencies 1, 0.1, 0.01, 0.001)
D_B34 = 8
OMEGA_B34 = 10000.0 ** (-np.arange(0, D_B34, 2) / D_B34)


def pe_b34(p, omega=OMEGA_B34):
    """Row p of the sinusoidal encoding: sin in the even columns, cos in the odd ones."""
    row = np.zeros(2 * len(omega))
    row[0::2], row[1::2] = np.sin(p * omega), np.cos(p * omega)
    return row


PAIR_B34 = np.array([0.6, 0.8])                                     # (PE[p, 2], PE[p, 3]) for some real position p


def shift_b34(pair, angle):
    """The shift matrix of the fiche (203.5) applied to a (sin, cos) pair; a negative angle shifts backwards."""
    return np.array([[math.cos(angle), math.sin(angle)], [-math.sin(angle), math.cos(angle)]]) @ pair


SHIFT_B34 = shift_b34(PAIR_B34, 10 * OMEGA_B34[1])                  # 10 positions on pair 1: 1 radian
PE_DOT_B34 = float(pe_b34(0) @ pe_b34(2))


# ✏️ B3.5: GPT-2 small (config.json of openai-community/gpt2) and the mini-GPT of the notebook
def gpt_params(n_layer, d, vocab, n_pos, tied=True):
    """Parameters of a GPT-2-style decoder: biases everywhere, F = 4d, learned positions, final layer norm."""
    attn, mlp, lns = 4 * d * d + 4 * d, 8 * d * d + 5 * d, 4 * d
    block = attn + mlp + lns
    return {"attn": attn, "mlp": mlp, "block": block, "blocks": n_layer * block, "wte": vocab * d,
            "wpe": n_pos * d, "ln_f": 2 * d,
            "total": n_layer * block + vocab * d * (1 if tied else 2) + n_pos * d + 2 * d,
            "non_embedding": n_layer * block + 2 * d}


GPT2_B35 = gpt_params(12, 768, 50257, 1024)
GPT2_UNTIED_B35 = gpt_params(12, 768, 50257, 1024, tied=False)["total"]
MINI_B35 = gpt_params(4, 128, 88, 128)
PAPER_B35 = 117e6                                                   # table 2 of the GPT-2 paper ("117M")

# quizzes and recalls with a computation
Q2C = (1 * 2 + 3 * 2) / math.sqrt(2)                                # q = (1, 3), k = (2, 2), d_k = 2
Q2D = 3 / (1 + 1 + 3)                                               # scores (0, 0, ln 3): exponentials 1, 1, 3
Q7E = 250 * 10000.0 ** (-2 * 16 / 64)                               # pair i = 16 of d_h = 64, position 250
Q8D = 2 * 512 * 2048 + 2048 + 512
Q11E = 2 * 2 * 100 * 64 * 4                                         # 2 (K and V) x layers x tokens x d x bytes
Q10E = math.log(65)
X_R3 = np.array([1.0, 2.0, 3.0, 6.0])
LN_R3 = (X_R3 - X_R3.mean()) / X_R3.std()                           # biased variance, as in your layer_norm

# The same answers from the libraries (no download): mylearn (reference), PyTorch, transformers
import torch
import torch.nn.functional as F
from torch import nn

A = mylearn.attention
# ✏️ B3.1 and B3.2: scaled_dot_product_attention of the reference and of PyTorch, with and without the masks
out, w = A.scaled_dot_product_attention(Q_B31, K_B31, V_B31)
assert np.allclose(w, A_B31) and np.allclose(out, O_B31)
tq, tk, tv = (torch.tensor(a) for a in (Q_B31, K_B31, V_B31))
assert np.allclose(F.scaled_dot_product_attention(tq, tk, tv).numpy(), O_B31)
assert np.allclose(A.scaled_dot_product_attention(Q_B31, K_B31, V_B31, scale=1.0)[1], A_NOSCALE_B31)
assert np.array_equal(A.causal_mask(3), CAUSAL_B32) and int(A.causal_mask(3).sum()) == 6
out_c, w_c = A.scaled_dot_product_attention(Q_B31, K_B31, V_B31, mask=A.causal_mask(3))
assert np.allclose(w_c, A_C_B32) and np.allclose(out_c, O_C_B32)
assert np.allclose(F.scaled_dot_product_attention(tq, tk, tv, is_causal=True).numpy(), O_C_B32)
assert np.array_equal(A.causal_mask(2, 3), BOTTOM_RIGHT_B32)
cache_out, _ = A.scaled_dot_product_attention(Q_B31[1:], K_B31, V_B31, mask=A.causal_mask(2, 3))
assert np.allclose(cache_out, O_C_B32[1:])                          # the KV cache gives rows 1 and 2 of the full run
top_left = F.scaled_dot_product_attention(tq[1:], tk, tv, is_causal=True).numpy()
assert not np.allclose(top_left, O_C_B32[1:])
assert np.allclose(top_left[0], tv[0].numpy())                      # the first new query sees only key 0
from torch.nn.attention.bias import causal_lower_right
assert np.allclose(F.scaled_dot_product_attention(tq[1:], tk, tv, attn_mask=causal_lower_right(2, 3)).numpy(),
                   O_C_B32[1:])
w_triu = A.scaled_dot_product_attention(Q_B31, K_B31, V_B31, mask=TRIU_B32)[1]
assert w_triu[2, 0] == 0 and w_triu[2, 1] == 0 and np.all(w_triu[0] > 0)

# ✏️ B3.3: nn.MultiheadAttention, split_heads and multi_head_attention of the reference, GQA with nn.Linear
mha = nn.MultiheadAttention(D_B33, H_B33)
assert sum(p.numel() for p in mha.parameters()) == MHA_B33 and tuple(mha.in_proj_weight.shape) == (3 * D_B33, D_B33)
mha12 = nn.MultiheadAttention(D_B33, 12)
assert sum(p.numel() for p in mha12.parameters()) == MHA_B33
rng = np.random.default_rng(0)
params = {f"W_{c}": rng.normal(size=(D_B33, D_B33)) / 20 for c in "qkvo"} | {f"b_{c}": np.zeros(D_B33) for c in "qkvo"}
x = rng.normal(size=(N_B33, T_B33, D_B33))
assert A.split_heads(x @ params["W_q"], H_B33).shape == (N_B33, H_B33, T_B33, DH_B33)
o, w = A.multi_head_attention(x, x, x, params, n_heads=H_B33)
assert o.shape == (N_B33, T_B33, D_B33) and w.shape == (N_B33, H_B33, T_B33, T_B33) and w.size == 2400
enc = rng.normal(size=(N_B33, S_B33, D_B33))
assert A.multi_head_attention(x, enc, enc, params, n_heads=H_B33)[1].shape == (N_B33, H_B33, T_B33, S_B33)
gqa = nn.ModuleList([nn.Linear(D_B33, D_B33), nn.Linear(D_B33, HKV_B33 * DH_B33), nn.Linear(D_B33, HKV_B33 * DH_B33),
                     nn.Linear(D_B33, D_B33)])
assert sum(p.numel() for p in gqa.parameters()) == GQA_B33

# ✏️ B3.4: positional_encoding of the reference; the shift matrix on real positions and on integer ones
PE = A.positional_encoding(200, D_B34)
assert np.allclose(PE[25], pe_b34(25)) and np.isclose(PE[0] @ PE[2], PE_DOT_B34) and np.isclose(PE[100] @ PE[102], PE_DOT_B34)
p_real = math.asin(0.6) / OMEGA_B34[1]                               # the position of the statement, about 6.4
assert np.allclose(pe_b34(p_real)[2:4], PAIR_B34) and np.allclose(pe_b34(p_real + 10)[2:4], SHIFT_B34)
for p in (0, 7, 33):
    assert np.allclose(shift_b34(PE[p, 2:4], 10 * OMEGA_B34[1]), PE[p + 10, 2:4])
assert round(2 * math.pi / OMEGA_B34[2]) == 628

# ✏️ B3.5: GPT-2 small and the mini-GPT, built without their weights (meta device)
from transformers import GPT2Config, GPT2LMHeadModel
with torch.device("meta"):
    gpt2 = GPT2LMHeadModel(GPT2Config())
    gpt2_untied = GPT2LMHeadModel(GPT2Config(tie_word_embeddings=False))
    mini = GPT2LMHeadModel(GPT2Config(n_layer=4, n_head=4, n_embd=128, n_positions=128, vocab_size=88,
                                       bos_token_id=0, eos_token_id=0))
count = lambda model: sum(p.numel() for p in model.parameters())   # noqa: E731
assert count(gpt2) == GPT2_B35["total"] and count(gpt2_untied) == GPT2_UNTIED_B35 and count(mini) == MINI_B35["total"]
assert count(gpt2.transformer.h[0]) == GPT2_B35["block"] and count(gpt2.transformer.h[0].attn) == GPT2_B35["attn"]
assert count(gpt2.transformer.h[0].mlp) == GPT2_B35["mlp"]
assert count(gpt2) - count(gpt2.transformer.wte) - count(gpt2.transformer.wpe) == GPT2_B35["non_embedding"]
block = nn.TransformerEncoderLayer(768, 12, 3072, dropout=0.0, activation="gelu", batch_first=True, norm_first=True)
assert count(block) == GPT2_B35["block"]

# quizzes and recalls with a computation
qa = torch.tensor([[1.0, 3.0]])
assert np.isclose((qa @ torch.tensor([[2.0], [2.0]])).item() / math.sqrt(2), Q2C)
assert np.isclose(torch.softmax(torch.tensor([0.0, 0.0, math.log(3)]), 0)[2].item(), Q2D)
assert int(A.causal_mask(5).sum()) == 15 and int(A.causal_mask(1, 9).sum()) == 9
angles = np.arange(251)[:, None] * 10000.0 ** (-2 * np.arange(32) / 64)
assert np.isclose(angles[250, 16], Q7E)
assert count(nn.Sequential(nn.Linear(512, 2048), nn.GELU(), nn.Linear(2048, 512))) == Q8D
assert count(nn.LayerNorm(1600)) == 3200 and count(nn.LayerNorm(768)) == 1536
assert np.allclose(mylearn.nn.regularization.layer_norm(X_R3[None], eps=0.0)[0], LN_R3)
assert np.allclose(F.layer_norm(torch.tensor(X_R3), (4,), eps=0.0).numpy(), LN_R3)
print("✅ the paper answers agree with mylearn (reference), PyTorch and transformers")

/root/venv313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ the paper answers agree with mylearn (reference), PyTorch and transformers


**B3.Q1 — Le goulot de l'encodeur-décodeur récurrent : pourquoi un seul vecteur ne suffit pas**

In [3]:
wb.record("B3.Q1a", "C", mistakes={"relis « Traduire avec deux RNN » (§203.1) : le décodeur de 2014 relit-il la source ?": "A", "relis « Traduire avec deux RNN » : par où passent les mots source ?": "B"})
wb.record("B3.Q1b", False, mistakes={"relis « Le goulot » (§203.1) : la taille de c dépend-elle de la phrase ?": True})
wb.record("B3.Q1c", 12 * 9, mistakes={"relis la question : on demande les poids de tous les pas de la traduction": 12,
          "relis la question : on demande les poids de tous les mots source, à chaque pas": 9,
          "tu as additionné : combien de poids à chaque pas, et combien de pas ?": 21})
wb.record("B3.Q1d", "A", mistakes={"relis l'étape 2 de l'idée de Bahdanau : sur quoi porte la softmax ?": "B", "relis le paragraphe qui suit les quatre étapes : l'alignement est-il un choix d'un seul mot ?": "C"})
wb.record("B3.Q1e", True, mistakes={"relis le paragraphe sur l'alignement souple (§203.1) : d'où viennent les poids ?": False})

WB_ANSWER {"hash": "365b1b487e54aca5cf2c3ec7bc0fae05ebf0de2f731ec8c9b5277f1b687ee8fd", "id": "B3.Q1a", "kind": "str", "mistakes": {"73a1c1f10cea604ed92b2ccd475ccb2920f1c345de1ff61e3900ffe211589bd2": "relis « Traduire avec deux RNN » (§203.1) : le décodeur de 2014 relit-il la source ?", "ab54e0550ba97f4b41c5c3ad3ba98e018daeee768648ee2c171e8e800ab88fa6": "relis « Traduire avec deux RNN » : par où passent les mots source ?"}}
📝 Ex B3.Q1a : réponse enregistrée (str).
WB_ANSWER {"hash": "3ddb75edcdb8230c49075e913c00f55d67248a78f57785530bd2516bfe551721", "id": "B3.Q1b", "kind": "bool", "mistakes": {"e119f29c4617b276bfc9116f17be9860ace156ddb4b5bebc18d1af1fe6641f51": "relis « Le goulot » (§203.1) : la taille de c dépend-elle de la phrase ?"}}
📝 Ex B3.Q1b : réponse enregistrée (bool).
WB_ANSWER {"hash": "d7abf0f2beea0c6bdbd61d5d335d6e932817b6f0c74b0231d587c0069aa4ed42", "id": "B3.Q1c", "kind": "int", "magnitude": 2, "mistakes": {"3520c9151bbc2628edb0ab185bd321076e616a8db8a4b8b76659192240a92d8e"

**B3.Q2 — Requête, clé, valeur : un dictionnaire où l'on cherche « à peu près »**

In [4]:
wb.record("B3.Q2a", "B", mistakes={"relis « Un dictionnaire où l'on cherche à peu près » : rend-on une seule valeur ?": "A", "relis le rôle de chaque vecteur : qu'est-ce que l'attention rend ?": "C"})
wb.record("B3.Q2b", False, mistakes={"relis « D'où viennent les requêtes, les clés et les valeurs » : combien de projections ?": True})
wb.record("B3.Q2c", Q2C, decimals=2, mistakes={"relis la question : le score est mis à l'échelle": 8.0,
          "on divise par la racine de d_k, pas par d_k": 4.0})
wb.record("B3.Q2d", Q2D, decimals=1, mistakes={"les scores passent par l'exponentielle avant d'être normalisés": 1.0,
          "compare les exponentielles des trois scores : sont-elles égales ?": 1 / 3})
wb.record("B3.Q2e", "C", mistakes={"c'est la forme des poids de la requête : on demande sa sortie": "A", "relis l'étape 5 de « La formule » : que devient la somme pondérée ?": "B"})

WB_ANSWER {"hash": "0c500beff3803fa26539287e6c1e91c08d5f9ab6ea877d80e92538792eb0a92e", "id": "B3.Q2a", "kind": "str", "mistakes": {"0c7dcacc226a8306ec9ddc885687313b2fc4cb7b058fc393bf2e3310eb616f6c": "relis le rôle de chaque vecteur : qu'est-ce que l'attention rend ?", "f1abd6fb46cae9266e705c0b27fb12e566f01efad966569ed890ed12cc923977": "relis « Un dictionnaire où l'on cherche à peu près » : rend-on une seule valeur ?"}}
📝 Ex B3.Q2a : réponse enregistrée (str).
WB_ANSWER {"hash": "9051f17cd9b1015c4874125fa0ee1c4764ba6e65241f1bc07f117d69be112246", "id": "B3.Q2b", "kind": "bool", "mistakes": {"e5f4dc3f52a51f62c09dbb8b6e946a92137a30742b93fb03d059872fe970021a": "relis « D'où viennent les requêtes, les clés et les valeurs » : combien de projections ?"}}
📝 Ex B3.Q2b : réponse enregistrée (bool).
WB_ANSWER {"decimals": 2, "hash": "f26ae514c28003ce5ab7eefb4873784df5350367135e8bfda01369fc153dadd6", "hash_coarse": "fddf848b8542f0dc54759a104ed9de7ff76234fe782cb0eecca001881399c872", "id": "B3.Q2c", 

**B3.Q3 — Pourquoi diviser les scores par √d_k ?**

In [5]:
wb.record("B3.Q3a", math.isqrt(256), mistakes={"c'est la variance : on demande l'écart-type": 256,
          "relis la question : d_k = 256": 8})
wb.record("B3.Q3b", True, mistakes={"relis « Pourquoi diviser par √d_k » : que vaut Var(aX) avec a = 1/√d_k ?": False})
wb.record("B3.Q3c", "A", mistakes={"relis le paragraphe sur la softmax saturée : quel poids prend le plus grand score ?": "B", "une softmax peut-elle donner un poids négatif ?": "C"})
wb.record("B3.Q3d", "C", mistakes={"l'échelle des scores compte avant l'exponentielle : relis l'effet de la variance des scores sur la softmax": "A", "diviser par plus que √d_k rend-il les scores plus grands ou plus petits ?": "B"})
wb.record("B3.Q3e", False, mistakes={"relis l'argument de la variance : de quels vecteurs le score est-il fait ?": True})

WB_ANSWER {"hash": "6c07c88d0007fa223235aa6b22cd53da8d94c8d109eae2ffa324494aa5ae5f86", "id": "B3.Q3a", "kind": "int", "magnitude": 1, "mistakes": {"9b3df75729c4f5d12a65ac1d8f2c6e0a45d199a0e7053b55b65615c4133a79b7": "c'est la variance : on demande l'écart-type", "a505cbfc6f6ee5372ac36e5bcaece471babc939fce04680c884d4daf4146ddf0": "relis la question : d_k = 256"}, "sign": 1}
📝 Ex B3.Q3a : réponse enregistrée (int).
WB_ANSWER {"hash": "97581c1b33d227f0e9dff24b0710a1c0fc68cc3f08ebb7b4ef6cd82aa12b2abd", "id": "B3.Q3b", "kind": "bool", "mistakes": {"eba65c48755501f1709c0051506a3c676e12fdfbd77cc04f7f38d443badccf75": "relis « Pourquoi diviser par √d_k » : que vaut Var(aX) avec a = 1/√d_k ?"}}
📝 Ex B3.Q3b : réponse enregistrée (bool).
WB_ANSWER {"hash": "eb84077f9b5e6608b379967715ce57e59c1f8816a13abc76db6b4b7776a3247e", "id": "B3.Q3c", "kind": "str", "mistakes": {"56c793590693963ba77efd1ad305753c560b374ae1fa5add32abcffbb0973ee1": "relis le paragraphe sur la softmax saturée : quel poids prend le 

**B3.Q4 — Masque causal : qui a le droit de regarder qui ?**

In [6]:
wb.record("B3.Q4a", int(np.tril(np.ones((5, 5))).sum()), mistakes={"compte à nouveau : la diagonale est-elle autorisée ?": 10,
          "le masque causal interdit des cases : lesquelles ?": 25,
          "compte toutes les cases vraies, pas seulement la diagonale": 5})
wb.record("B3.Q4b", "B", mistakes={"relis la phrase sous la formule du masque causal : combien de cases vraies a la ligne i ?": "A", "le masque causal cache-t-il le futur ou le passé ?": "C"})
wb.record("B3.Q4c", False, mistakes={"relis la conséquence utile du masque causal (§203.3)": True})
wb.record("B3.Q4d", 9, mistakes={"c'est l'alignement en haut à gauche de is_causal=True : relis celui de causal_mask": 1,
          "la dernière requête voit-elle sa propre clé ?": 8})
wb.record("B3.Q4e", "A", mistakes={"c'est le rôle d'un autre masque : relis « Le masque de padding »": "B", "relis le rôle du masque : change-t-il le nombre de scores calculés ?": "C"})

WB_ANSWER {"hash": "9839be5b6a783fdfda4cce2b4bbb6d4b922ef80bcb067f81d845060b8e1cda6a", "id": "B3.Q4a", "kind": "int", "magnitude": 1, "mistakes": {"3132e5ebc19f1e886f799451824770cd8835f11da34f9a0611eceef7fe927950": "compte toutes les cases vraies, pas seulement la diagonale", "60a96eed531c481cbe67659c148668f1ed19b73ab9f2c0cc744e55d1194e033d": "compte à nouveau : la diagonale est-elle autorisée ?", "9689744674a37c1e5f9a3281aa00e3729eb459592225f6971fd8f6d37532a12c": "le masque causal interdit des cases : lesquelles ?"}, "sign": 1}
📝 Ex B3.Q4a : réponse enregistrée (int).
WB_ANSWER {"hash": "666cd8f9db6676a3965e4431926db963468fb9b72d2ca40aff154b85b85583e4", "id": "B3.Q4b", "kind": "str", "mistakes": {"ad1f60da5fe4aa189a33b9591b9ffa0fadaef1601dda2c62fef683f006347832": "le masque causal cache-t-il le futur ou le passé ?", "c8cf0d076c34609364dc3000efacaf238350cb5a46166fdc0c022d1f6eea9a16": "relis la phrase sous la formule du masque causal : combien de cases vraies a la ligne i ?"}}
📝 Ex B3.Q

**B3.Q5 — Auto-attention ou attention croisée : d'où viennent Q, K et V ?**

In [7]:
wb.record("B3.Q5a", "A", mistakes={"relis la formule de l'attention croisée (§203.3) : d'où vient Q ?": "B", "relis « Auto-attention et attention croisée » : les clés et les requêtes viennent-elles de la même séquence ?": "C"})
wb.record("B3.Q5b", "(7, 12)", mistakes={"relis l'ordre des axes de Q Kᵀ : les lignes sont-elles les requêtes ou les clés ?": "(12, 7)",
          "les clés viennent de la source : combien de colonnes ?": "(7, 7)",
          "les requêtes viennent de la traduction : combien de lignes ?": "(12, 12)"})
wb.record("B3.Q5c", False, mistakes={"relis le tableau des familles (§203.7) : quel masque dans l'attention croisée ?": True})
wb.record("B3.Q5d", "C", mistakes={"les trois arguments ne sont pas les mêmes : relis l'ordre query, key, value": "A", "le premier argument fournit les requêtes : qui regarde qui ?": "B"})
wb.record("B3.Q5e", True, mistakes={"relis la définition de l'auto-attention (§203.3)": False})

WB_ANSWER {"hash": "440d71bb2da4e81d2c7e5528f0c5044a0048deff4a2626128415941f2f00ce1a", "id": "B3.Q5a", "kind": "str", "mistakes": {"44ca82a3c672efec6fc3ee954fe600944f01f6e81a14b980460ed8d6063e8ca2": "relis la formule de l'attention croisée (§203.3) : d'où vient Q ?", "e4551531ecc70971092580185f905aba096d1298034acd012d97110e46ef6c4b": "relis « Auto-attention et attention croisée » : les clés et les requêtes viennent-elles de la même séquence ?"}}
📝 Ex B3.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "c914bea10f90373a4066bc7124c71826035e7ade23fe52dc8c50795f3aab9ea8", "id": "B3.Q5b", "kind": "str", "mistakes": {"1818a5b05f9916ad76df432839fdc340ded42e06848eceb33b639925d6fdf5c4": "les clés viennent de la source : combien de colonnes ?", "481632a6d89e4d618fffa66d61df9a35455d8e65bcd6650c8f03b2d40a156b24": "relis l'ordre des axes de Q Kᵀ : les lignes sont-elles les requêtes ou les clés ?", "b41a95a606f4efe97e504da65547154c1c1d193c12182e66975daba6755cc7b8": "les requêtes viennent de la tr

**B3.Q6 — Plusieurs têtes pour le prix d'une : que gagne-t-on ?**

In [8]:
wb.record("B3.Q6a", 512 // 16, mistakes={"tu as multiplié : relis la définition de d_h": 8192,
          "recompte avec H = 16": 64})
wb.record("B3.Q6b", False, mistakes={"relis « Compter les paramètres » (§203.4)": True})
wb.record("B3.Q6c", 32 // 4, mistakes={"c'est le nombre de têtes de clés : on demande un rapport": 4,
          "on demande un rapport, pas une différence": 28,
          "on demande un rapport, pas un produit": 128})
wb.record("B3.Q6d", "A", mistakes={"relis le piège classique « un reshape direct » : y a-t-il une erreur ?": "B", "relis le piège classique « un reshape direct » : quelle tête reçoit quelles features ?": "C"})
wb.record("B3.Q6e", True, mistakes={"relis le paragraphe sur repeat_interleave (§203.4), et écris les deux ordres de têtes": False})

WB_ANSWER {"hash": "e4ed72b7f58fa2743f617ec39a1ed0783ef3c3091ac8fb7e6f32bf7e603d3ce5", "id": "B3.Q6a", "kind": "int", "magnitude": 1, "mistakes": {"c2957c0ba0bb9350221b537703d8fbcaa63fb277017e37ee08465487b8e27964": "tu as multiplié : relis la définition de d_h", "d061fe56da40a9e284464b94cff329cc03755d204e87a3715bfaadb3de9f7158": "recompte avec H = 16"}, "sign": 1}
📝 Ex B3.Q6a : réponse enregistrée (int).
WB_ANSWER {"hash": "d2fdd9346b278420f6e8ec6c8d2ee46c010b799c0f138ad5ff1b253838a2f308", "id": "B3.Q6b", "kind": "bool", "mistakes": {"9518d6e5e731728c0b299e5daa5f5b8f1cb17036cc36e128a20fa97b834b682e": "relis « Compter les paramètres » (§203.4)"}}
📝 Ex B3.Q6b : réponse enregistrée (bool).
WB_ANSWER {"hash": "0056816709584d98862e499677824a6d63aed536c242aa3146ece728c81d6d06", "id": "B3.Q6c", "kind": "int", "magnitude": 0, "mistakes": {"009c83cc2be8d0241958e747a9e18d39a315264f2f189afbffd0a784e65c78af": "on demande un rapport, pas une différence", "1b633a9f60ee41d34a8b5d299ec89a0f1a1762779ae

**B3.Q7 — Sans encodage de position, l'attention voit-elle l'ordre des mots ?**

In [9]:
wb.record("B3.Q7a", True, mistakes={"relis « L'attention ne voit pas l'ordre » (§203.5)": False})
wb.record("B3.Q7b", "A", mistakes={"sans position, l'attention sait-elle qui est voisin de qui ?": "B", "la sortie est une moyenne de valeurs : pourquoi serait-elle nulle ?": "C"})
wb.record("B3.Q7c", "B", mistakes={"relis « Les positions apprises » : quelle est la limite de la table ?": "A", "relis « Les positions apprises » : la table a-t-elle une règle pour les positions trop grandes ?": "C"})
wb.record("B3.Q7d", "C", mistakes={"relis le premier paragraphe de « RoPE » : touche-t-il aux embeddings ?": "A", "relis les trois points pratiques de RoPE : quels vecteurs sont tournés ?": "B"})
wb.record("B3.Q7e", Q7E, decimals=1, mistakes={"vérifie l'exposant de la fréquence θ_i (fiche, « RoPE »)": 250 * 10000.0 ** (-16 / 64),
          "relis la question : la paire i = 16, pas la paire 0": 250.0,
          "on demande des radians, pas des degrés": math.degrees(Q7E)})

WB_ANSWER {"hash": "0182f00325cca3fdf5a6f5fce5c8b5f550b49e90548e713bcabed6c729c3394a", "id": "B3.Q7a", "kind": "bool", "mistakes": {"3c65f6e11e3bb55e6464779c0ba82458e517bcfb45e05d9adb3fa88080688c42": "relis « L'attention ne voit pas l'ordre » (§203.5)"}}
📝 Ex B3.Q7a : réponse enregistrée (bool).
WB_ANSWER {"hash": "f059562f81f7cf23e2da88c3da194075a8f73a23444cc07d8fc427bb361bb0d5", "id": "B3.Q7b", "kind": "str", "mistakes": {"0626fe81c32836a84500c5e0465cac18b2e0c12d375abe415c2d5d87ee7e72b1": "la sortie est une moyenne de valeurs : pourquoi serait-elle nulle ?", "11e09065927cd826c5b795fe514e6d592944788e740efc99fe3306a674fd4a74": "sans position, l'attention sait-elle qui est voisin de qui ?"}}
📝 Ex B3.Q7b : réponse enregistrée (str).
WB_ANSWER {"hash": "f75e68ac8a038364eb058bad47ed1475e165f0f64100a5171e28aa58d56edbb6", "id": "B3.Q7c", "kind": "str", "mistakes": {"12b2b18e45787106b73c05a2fdb04ca4a53d818e6ab29632b7a05545192eae66": "relis « Les positions apprises » : la table a-t-elle une rè

**B3.Q8 — Résiduel, pré-LN ou post-LN : ce qui rend un Transformer profond entraînable**

In [10]:
wb.record("B3.Q8a", "C", mistakes={"relis les deux formules du §203.6 : où se place la layer norm en pré-LN ?": "A", "relis les deux formules du §203.6 : que manque-t-il ?": "B"})
wb.record("B3.Q8b", True, mistakes={"relis le paragraphe qui suit la figure des deux blocs (§203.6)": False})
wb.record("B3.Q8c", "A", mistakes={"relis le paragraphe sur Xiong et coll. : où les gradients sont-ils grands ?": "B", "relis le paragraphe sur Xiong et coll. : est-ce le pré-LN ou le post-LN qui supporte un warmup court ?": "C"})
wb.record("B3.Q8d", Q8D, mistakes={"n'oublie pas les biais des deux couches denses": 2 * 512 * 2048,
          "vérifie la taille du biais de la seconde couche": 2 * 512 * 2048 + 2 * 2048,
          "vérifie la taille du biais de la première couche": 2 * 512 * 2048 + 2 * 512,
          "le MLP a deux couches denses": 512 * 2048 + 2048})
wb.record("B3.Q8e", False, mistakes={"relis le rôle de la connexion résiduelle (§203.6 et le rappel R3)": True})

WB_ANSWER {"hash": "45886c28be3a8207f1eeb4613880063100025b49c84f0a6b0d3d5372488ee7af", "id": "B3.Q8a", "kind": "str", "mistakes": {"56fc085d3f792c61409caa89bdafecbe3f6038011bde72d2c443e342837e33ad": "relis les deux formules du §203.6 : que manque-t-il ?", "5d4441a8b32e1c4db67b4c5af94338d1a02107b319b84402e058cd2fba633613": "relis les deux formules du §203.6 : où se place la layer norm en pré-LN ?"}}
📝 Ex B3.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "17637d9fa32431c85a83ef970d2cecfaca4f0f77f349226c2a4bc12a93fc401d", "id": "B3.Q8b", "kind": "bool", "mistakes": {"5cc0ca6d91067c7ecb58e9c7a357f13555de58f5e95e878a6ee699d8690acac5": "relis le paragraphe qui suit la figure des deux blocs (§203.6)"}}
📝 Ex B3.Q8b : réponse enregistrée (bool).
WB_ANSWER {"hash": "ebe136b95ee395789dbbd9ef4b8b44a5c0a0ef2fdf8193123947b03085c88c23", "id": "B3.Q8c", "kind": "str", "mistakes": {"973798d3d3f2c947856ad1d837b883f11691e7e10326fb65b132bb175b4bb4e0": "relis le paragraphe sur Xiong et coll. : est-ce 

**B3.Q9 — BERT, GPT, T5 : quel masque, quel objectif, quel usage ?**

In [11]:
wb.record("B3.Q9a", "A", mistakes={"c'est l'objectif d'un décodeur : relis « L'encodeur : BERT »": "B", "c'est l'objectif de T5 : relis « L'encodeur : BERT »": "C"})
wb.record("B3.Q9b", 200 * 15 // 100, mistakes={"relis la question : 15 % de 200 tokens": 15,
          "relis la question : on demande toutes les positions choisies": 24,
          "on demande les positions choisies, pas les autres": 170})
wb.record("B3.Q9c", 30 * 80 // 100, mistakes={"relis la question : on demande celles qui deviennent [MASK]": 30,
          "relis la question : parmi les positions choisies seulement": 160,
          "relis la proportion des positions choisies qui deviennent [MASK] (§203.7)": 3})
wb.record("B3.Q9d", True, mistakes={"relis le paragraphe sur le décodeur (§203.7) : que permet le prompt ?": False})
wb.record("B3.Q9e", "B", mistakes={"relis la ligne « Sortie typique » du tableau des familles": "A", "relis la ligne « Sortie typique » du tableau des familles : que produit cette famille ?": "C"})

WB_ANSWER {"hash": "962f3fadbf1669a147e19bd49af05e976b8a3bfd72014cfb7770d0113780413b", "id": "B3.Q9a", "kind": "str", "mistakes": {"87a271917067ac33962d603c84842b41fe8e288ebe4fee195124bb2f40d66dfb": "c'est l'objectif d'un décodeur : relis « L'encodeur : BERT »", "90eb2b7abb12c2a20747ba284c91003bb8cb6c793c29e6912e7397f8f10d29b2": "c'est l'objectif de T5 : relis « L'encodeur : BERT »"}}
📝 Ex B3.Q9a : réponse enregistrée (str).
WB_ANSWER {"hash": "df06b237e2995a65c24e0d4060301addb962e52a712d16f7305e95e9fe1eb7e0", "id": "B3.Q9b", "kind": "int", "magnitude": 1, "mistakes": {"1079840f3772e7a5f70dcf1c85cb728a69fc0d64c353414be304f05b5026f293": "relis la question : on demande toutes les positions choisies", "dfc59f1597819b52db397728a354c7ea78348230680def5647fbd6b3f52f855b": "on demande les positions choisies, pas les autres", "f6ab33e64c2a1bc4e611a6280849e19b5016efbbc0cfadd72a764efae8c2f4f9": "relis la question : 15 % de 200 tokens"}, "sign": 1}
📝 Ex B3.Q9b : réponse enregistrée (int).
WB_ANSWE

**B3.Q10 — Décalage d'un token : ce que chaque position apprend à prédire**

In [12]:
wb.record("B3.Q10a", "B", mistakes={"les cibles sont-elles les entrées elles-mêmes ? Relis le tableau « chat » (§203.7)": "A", "relis le tableau « chat » : combien de cibles pour T entrées ?": "C"})
wb.record("B3.Q10b", 256, mistakes={"relis le tableau « chat » : la dernière position a-t-elle une cible ?": 255,
          "relis la question : T = 256": 128})
wb.record("B3.Q10c", False, mistakes={"relis la définition du teacher forcing (§203.7) : quelle séquence est donnée en entrée ?": True})
wb.record("B3.Q10d", "C", mistakes={"le modèle voit-il la cible dans son entrée ?": "A", "une loss qui diverge vient d'un learning rate ou d'une initialisation : ici, que voit le modèle ?": "B"})
wb.record("B3.Q10e", Q10E, decimals=2, mistakes={"le logarithme est naturel ici, pas en base 2": math.log2(65),
          "le logarithme est naturel ici, pas décimal": math.log10(65),
          "on demande une loss, pas une probabilité": 1 / 65})

WB_ANSWER {"hash": "56ea0a67e7236628985ccd2d70bd1f9670ef28f5fc699057fb6ebf3fc8bbab8e", "id": "B3.Q10a", "kind": "str", "mistakes": {"028a97df08d2655767e02d3d270ce91db85fc3d8d5d8524db0cf57ec4034bc2b": "relis le tableau « chat » : combien de cibles pour T entrées ?", "3016becffd80dad8fe684d76f997e127ccb734f55094795ed5c846a71fc88f9e": "les cibles sont-elles les entrées elles-mêmes ? Relis le tableau « chat » (§203.7)"}}
📝 Ex B3.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "d45bf93dab3ac9f6f9c9fddc7220053b39cc80f23ffc9bb1ebf093e6a2c63b50", "id": "B3.Q10b", "kind": "int", "magnitude": 2, "mistakes": {"62604539a0b80e85947c700f8f6369dff35694cb6cfbea0ef30e973563ca8a5f": "relis la question : T = 256", "cf144cf202fae0d8e3b467919d34721fa7fe91bc5d093a7b0d67ef28fbad8f99": "relis le tableau « chat » : la dernière position a-t-elle une cible ?"}, "sign": 1}
📝 Ex B3.Q10b : réponse enregistrée (int).
WB_ANSWER {"hash": "5e6646c614f426860a1bd0451f4eb5614a877d0b7fd71ba34a4da5280b3437a3", "id": "B

**B3.Q11 — KV cache : ce que l'on garde et pourquoi la génération s'accélère**

In [13]:
wb.record("B3.Q11a", "B", mistakes={"relis « Le KV cache » : les anciennes requêtes resservent-elles ?": "A", "relis « Le KV cache » : que garde-t-on pour chaque couche ?": "C"})
wb.record("B3.Q11b", 1, mistakes={"le nouveau token a-t-il besoin d'une requête ?": 0,
          "relis l'étape 1 du KV cache : combien de tokens passent dans le modèle ?": 2})
wb.record("B3.Q11c", False, mistakes={"relis la vérification de la fiche (une couche, cache contre calcul complet) : les sorties diffèrent-elles ?": True})
wb.record("B3.Q11d", "A", mistakes={"relis « Ce que coûte le cache » : de quoi dépend la formule ?": "B", "relis la formule du cache : T y apparaît-il au carré ?": "C"})
wb.record("B3.Q11e", Q11E, mistakes={"recompte les facteurs de la formule du cache : il en manque un de 2": Q11E // 2,
          "tu as compté des valeurs, ou oublié deux facteurs : relis la formule du cache": Q11E // 4,
          "on demande des octets, pas des bits": Q11E * 8})

WB_ANSWER {"hash": "1ec2c8b572721373e9b675f4aa3438ca753778e6ac9ec61cca785ff68be09763", "id": "B3.Q11a", "kind": "str", "mistakes": {"965cc3d33795329febc0d432a521c0fe281632351b8ed5a8f1afdda9fcf52235": "relis « Le KV cache » : que garde-t-on pour chaque couche ?", "d79ce9dce63dea0ec0a92e97b45e4f2d4efe9143d2cf43d307a9a70b88cde1e1": "relis « Le KV cache » : les anciennes requêtes resservent-elles ?"}}
📝 Ex B3.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash": "ed1001461a68f5775522f7bb633eb9dd3fae0ec8430a96792566ccb1b14b0b19", "id": "B3.Q11b", "kind": "int", "magnitude": 0, "mistakes": {"8b835c077d48090ae19524c14de85be7a63e3543cdadfe2e2aa5ae5b31c75d90": "le nouveau token a-t-il besoin d'une requête ?", "ff75e62df75242d6efdd680d006e2072e64100f227c20f190a23d4af2caa24c0": "relis l'étape 1 du KV cache : combien de tokens passent dans le modèle ?"}, "sign": 1}
📝 Ex B3.Q11b : réponse enregistrée (int).
WB_ANSWER {"hash": "920d656e3b3a9e0ecf6d9b83b5371c71445d68b1e5646ede70aeb479333f67ea", "id": 

**B3.Q12 — Coût quadratique du contexte et ce que change FlashAttention**

In [14]:
wb.record("B3.Q12a", 8 ** 2, mistakes={"relis « La mémoire : la matrice T × T » : comment la mémoire croît-elle avec T ?": 8,
          "le carré de la longueur, pas son cube": 512,
          "relis la question : on multiplie la longueur par 8": 4})
wb.record("B3.Q12b", False, mistakes={"relis le résultat de FlashAttention (§203.10) : est-ce une approximation ?": True})
wb.record("B3.Q12c", "C", mistakes={"FlashAttention change-t-il le modèle ?": "A", "relis le résultat de FlashAttention : que devient le nombre de FLOPs ?": "B"})
wb.record("B3.Q12d", 12 * 768, mistakes={"le seuil n'est pas T = d : relis le rapport des deux coûts (§203.10)": 768,
          "relis la question : un décodeur causal, où une requête voit en moyenne la moitié des clés": 24 * 768})
wb.record("B3.Q12e", True, mistakes={"relis l'exemple de la fiche pour d = 4 096 (§203.10)": False})

WB_ANSWER {"hash": "bbd944b276917a29fd63980d97e60c812831b11a983dbaf05abf8337c32eae88", "id": "B3.Q12a", "kind": "int", "magnitude": 1, "mistakes": {"43d6d893f3bcbb8b9b7b4dd09c5790a80ad16d0f93d031f461eb905d28dea56d": "relis « La mémoire : la matrice T × T » : comment la mémoire croît-elle avec T ?", "4478c05c3d991d22a3cc25f98250d1bb560d75266b6340f3f5e8e7d6ce4f642f": "le carré de la longueur, pas son cube", "838e4ea115e62c94fe95ac0a7fbbfb58c84a0cae3864afa41cbe81768d2035fb": "relis la question : on multiplie la longueur par 8"}, "sign": 1}
📝 Ex B3.Q12a : réponse enregistrée (int).
WB_ANSWER {"hash": "f01a82525a968d083e622eeaefc8e5d11ba05ec85e37c229771a98056e0743c8", "id": "B3.Q12b", "kind": "bool", "mistakes": {"ec2977328ae1ff3280ebc6ec1c1d7823c85c325dbd9b736807ca8e6d3eee184f": "relis le résultat de FlashAttention (§203.10) : est-ce une approximation ?"}}
📝 Ex B3.Q12b : réponse enregistrée (bool).
WB_ANSWER {"hash": "1b3b4007a5bc4a7faed0639c9d8303dff5dd8a6be5f3384aa0152fa046eb8abb", "id":

**B3.R1 — De quoi dépend le nombre de tokens d'un texte ?**

In [15]:
wb.record("B3.R1a", False, mistakes={"compare le nombre de tokens d'un même texte avec deux tokenizers de B2": True})
wb.record("B3.R1b", 4000 * 10 // 32, mistakes={"relis la définition de la compression (B2) : des octets par token": 12800},
          fractional="on demande un nombre de tokens")
wb.record("B3.R1c", "C", mistakes={"relis B2 (§202.3) : que fait une fusion au nombre de tokens du corpus ?": "A", "relis B2 : une fusion de plus change-t-elle le découpage ?": "B"})
wb.record("B3.R1d", True, mistakes={"relis la fertilité du français avec GPT-2 (B2, §202.6)": False})
wb.record("B3.R1e", "B", mistakes={"relis la compression de B2 : combien de caractères par token BPE ?": "A", "compare la longueur moyenne d'un token BPE à celle d'un caractère": "C"})

WB_ANSWER {"hash": "59c0763509f21f9c4a68d8b1cca63563f9cefcd918d269fd48835db4bb6047a7", "id": "B3.R1a", "kind": "bool", "mistakes": {"cc627f1709012568f57d72428a604c2b06b23c696c14e038a924c590a4f46add": "compare le nombre de tokens d'un même texte avec deux tokenizers de B2"}}
📝 Ex B3.R1a : réponse enregistrée (bool).
WB_ANSWER {"fractional": "on demande un nombre de tokens", "hash": "6796e35c310f289bf3560eadb8c9d6a2d89ba74f443573c247c84de309e3b3e7", "id": "B3.R1b", "kind": "int", "magnitude": 3, "mistakes": {"464367ad85fa9aef0ca1ce84e4e8d1b064c440816625eae9e164567f080c5d51": "relis la définition de la compression (B2) : des octets par token"}, "sign": 1}
📝 Ex B3.R1b : réponse enregistrée (int).
WB_ANSWER {"hash": "3cdb0fba40d133ab77462b3091f8c9589ad1b27f4c327f7dd30355a6688faabc", "id": "B3.R1c", "kind": "str", "mistakes": {"6fd00b25201962d83e67f8b50ab1bffdebb4535d41d7ba104b8ce6e4e3f11796": "relis B2 : une fusion de plus change-t-elle le découpage ?", "dc5a1dfd0c6ad3991ce59712bda72f3eb722

**B3.R2 — Pourquoi le gradient s'évanouit dans un RNN déroulé**

In [16]:
wb.record("B3.R2a", 0.9 ** 50, decimals=4, mistakes={"relis le ch. 22 : le gradient traverse 50 jacobiennes l'une après l'autre": 0.9 * 50,
          "relis la question : 50 pas": 0.9})
wb.record("B3.R2b", False, mistakes={"relis le ch. 22 : le LSTM atténue-t-il ou supprime-t-il le problème ?": True})
wb.record("B3.R2c", "B", mistakes={"c'est le chemin dans un RNN : relis le tableau de §203.1": "A", "relis le tableau de §203.1 : chemin entre deux tokens éloignés": "C"})
wb.record("B3.R2d", "A", mistakes={"le dropout régularise : relis ce que le ch. 22 fait contre l'explosion": "B", "la température sert à la génération": "C"})
wb.record("B3.R2e", False, mistakes={"relis « Et si l'on supprimait les RNN ? » (§203.1)": True})

WB_ANSWER {"decimals": 4, "hash": "d01673cc756acaa65177ed5907be0e2fc923bd3484f5470e5597a790283b9b39", "hash_coarse": "ac1d905d7141a7f7101250805311d42f911c3d3db16329c932cdacc94a2e5d7f", "id": "B3.R2a", "kind": "float", "magnitude": -3, "mistakes": {"e7a94e481af42ab178e3c13b45fccb8195d595cd0a19983eadda254f803297e5": "relis le ch. 22 : le gradient traverse 50 jacobiennes l'une après l'autre", "f414f86325b8d045c67c71470c690711e3ecec4d239b7f4e8224322ecf4ea3c1": "relis la question : 50 pas"}, "sign": 1}
📝 Ex B3.R2a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"hash": "194c9dba34f462f742a0cadad827631dc71695904690d6dfe3977c40ad39ba4a", "id": "B3.R2b", "kind": "bool", "mistakes": {"f6823f60b937f887e0749095bc8f7a4f9fb6238a024a88e97ed33ddcc48b11f2": "relis le ch. 22 : le LSTM atténue-t-il ou supprime-t-il le problème ?"}}
📝 Ex B3.R2b : réponse enregistrée (bool).
WB_ANSWER {"hash": "f0bc7706b2de96ff72b659d0a809f5ade5ae0c52f09c403ad128cd1893f090e7", "id": "B3.R2c", "kind": "str", "mist

**B3.R3 — Layer norm et connexion résiduelle, à quoi sert chacune ?**

In [17]:
wb.record("B3.R3a", LN_R3[0], decimals=3, mistakes={"tu as divisé par la variance : relis la formule de ta layer_norm": -2 / X_R3.var(),
          "relis la docstring de ta layer_norm : quelle variance utilise-t-elle ?": -2 / X_R3.std(ddof=1),
          "vérifie le signe": -LN_R3[0],
          "as-tu soustrait la moyenne ?": 1 / X_R3.std()})
wb.record("B3.R3b", False, mistakes={"relis le ch. 20 : sur quel axe la layer norm calcule-t-elle ses statistiques ?": True})
wb.record("B3.R3c", "B", mistakes={"dérive y terme par terme : que donne le premier terme ?": "A", "dérive y terme par terme : que donne le second terme ?": "C"})
wb.record("B3.R3d", True, mistakes={"relis le 🔗 Lien avec le ch. 20 (§203.6)": False})
wb.record("B3.R3e", 2 * 1600, mistakes={"relis le ch. 20 : combien de vecteurs de paramètres a une layer norm ?": 1600,
          "relis : γ et β sont-ils des matrices ?": 2 * 1600 ** 2})

WB_ANSWER {"decimals": 3, "hash": "4591a52e970eaac97a28508e67842e184ec36c522951d9d44d26306aa1a2eef8", "hash_coarse": "08f83ad8f3243a300f63c593664789741b6ee3709de592a35a2238e7fab80473", "id": "B3.R3a", "kind": "float", "magnitude": 0, "mistakes": {"3bd45f027fde0711dd8d54e3a5aafd2363e5c85c6503ddcd7953fbecfa7aa8ef": "relis la docstring de ta layer_norm : quelle variance utilise-t-elle ?", "5fa767f52bc018309cd3eaa6863861aa4752e07b5af6ee7b34988db970bbd8df": "vérifie le signe", "6d95418a22766ec985df5438a6e04fa8226bc51826b2f60a768ced0f801586fd": "tu as divisé par la variance : relis la formule de ta layer_norm", "8830fbdb3c4f0055439a8e04a5412c76b96046c0145f769af8c40081e9ed4e6d": "as-tu soustrait la moyenne ?"}, "sign": -1}
📝 Ex B3.R3a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"hash": "c96219725e488ea484f13ce2a19851ac47976932279936f8e8be042e5226a419", "id": "B3.R3b", "kind": "bool", "mistakes": {"1be3a179cce597aee0ebebfb644c96cd2021c9f92ff3ba5e0d99b76b1f75ca00": "relis le ch. 20

**Ex B3.1 — L'attention à la main : trois tokens, d_k = 2, scores, softmax et sortie**

In [18]:
wb.record("B3.1a", Q_B31[2].tolist(), mistakes={"les tokens sont numérotés à partir de 0 : quelle ligne de X est x_2 ?": Q_B31[1].tolist(),
          "relis la question : on demande la requête, avec W_Q": K_B31[2].tolist()})
wb.record("B3.1b", S_B31[2, 1], decimals=3, mistakes={"relis la définition de s_ij dans l'énoncé : le score est mis à l'échelle": RAW_B31[2, 1],
          "on divise par la racine de d_k, pas par d_k": RAW_B31[2, 1] / 2,
          "vérifie la clé : les tokens sont numérotés à partir de 0": S_B31[2, 2],
          "c'est le score de la requête 1 pour la clé 2 : relis l'ordre des indices": S_B31[1, 2]})
wb.record("B3.1c", A_B31[2, 1], decimals=3, mistakes={"relis la définition de s_ij : la division par √d_k": A_NOSCALE_B31[2, 1],
          "la softmax porte sur les clés (chaque ligne), pas sur les requêtes": A_COLUMNS_B31[2, 1],
          "vérifie la clé : les tokens sont numérotés à partir de 0": A_B31[2, 2],
          "c'est le poids de la requête 1 pour la clé 2 : relis l'ordre des indices": A_B31[1, 2]})
wb.record("B3.1d", O_B31[2].tolist(), decimals=3, mistakes={"relis la définition de s_ij : la division par √d_k": (A_NOSCALE_B31[2] @ V_B31).tolist(),
          "la sortie moyenne les valeurs, pas les clés": (A_B31[2] @ K_B31).tolist(),
          "la softmax porte sur les clés (chaque ligne), pas sur les requêtes": (A_COLUMNS_B31[2] @ V_B31).tolist()})
wb.record("B3.1e", A_NOSCALE_B31[2, 2], decimals=3, mistakes={"relis la question : ici, sans la division par √d_k": A_B31[2, 2],
          "la softmax porte sur les clés (chaque ligne), pas sur les requêtes": softmax_rows(RAW_B31.T).T[2, 2]})
wb.record("B3.1f", True, mistakes={"compare les scores des clés 1 et 2 pour la requête 0": False})
wb.record("B3.1g", float(A_B31[:, 0].sum()), decimals=3, mistakes={"relis la question : on additionne la colonne 0, sur les trois requêtes": 1.0,
          "relis la définition de s_ij : la division par √d_k": float(A_NOSCALE_B31[:, 0].sum())})

WB_ANSWER {"decimals": 0, "element_hashes": ["5886e1852c91bb1856617fd8444c33a84861c5a5127657519ef9138f88a59514", "649c94e2800cacd17adfecfc9359a292d4e6ae5fcaf6f8abd9de98d9c450bce6"], "hash": "0f77249c0b243fdfc63a6783c60d68399ccc67071fcac8e24b52e13ce8228564", "id": "B3.1a", "integer": true, "kind": "array", "mistakes": {"1fcbe6882b2758a914ee3d66cab7a06c125184a21af620c5692b4375c4ca7899": "les tokens sont numérotés à partir de 0 : quelle ligne de X est x_2 ?", "f5a79ac11079606dd3774f89c436eae443aa690bb69f133e60f8aa47ce1476d3": "relis la question : on demande la requête, avec W_Q"}, "shape": [2]}
📝 Ex B3.1a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "c7b5bf59877aacb16dfa0b24779ac800029812e6bcb7d7b7689de9471bb2ddf0", "hash_coarse": "5abf9eb253a5328d30fbe2958119a0092faf43ce14e917250f132aecb519f21e", "id": "B3.1b", "kind": "float", "magnitude": 0, "mistakes": {"1f20f7d5d4f4ec70e4dfe2d0f416e3a8a5ed6fb3d234d65bf0c2d9a1ae75580d": "on divise par la racine de d_

**Ex B3.2 — Le même calcul avec un masque causal**

In [19]:
wb.record("B3.2a", int(CAUSAL_B32.sum()), mistakes={"la diagonale est-elle autorisée ?": 3,
          "le masque causal interdit des cases : lesquelles ?": 9})
wb.record("B3.2b", A_C_B32[1].tolist(), decimals=3, mistakes={"un masque ne se contente pas de mettre des poids à 0 : relis l'exemple chiffré du §203.3": A_ZERO_B32[1].tolist(),
          "relis la définition de s_ij : la division par √d_k": A_NOSCALE_C_B32[1].tolist(),
          "relis la ligne 1 du masque causal : quelles clés sont autorisées ?": A_TRIU_B32[1].tolist(),
          "relis la question : avec le masque": A_B31[1].tolist()})
wb.record("B3.2c", O_C_B32[1].tolist(), decimals=3, mistakes={"les poids de la ligne masquée font-ils 1 ? Relis l'exemple chiffré du §203.3": (A_ZERO_B32[1] @ V_B31).tolist(),
          "relis la question : avec le masque": O_B31[1].tolist(),
          "relis la ligne 1 du masque causal : quelles clés sont autorisées ?": (A_TRIU_B32[1] @ V_B31).tolist()})
wb.record("B3.2d", O_C_B32[0].tolist(), decimals=3, mistakes={"relis la question : avec le masque": O_B31[0].tolist(),
          "un masque ne se contente pas de mettre des poids à 0 : relis l'exemple chiffré du §203.3": (A_ZERO_B32[0] @ V_B31).tolist()})
wb.record("B3.2e", True, mistakes={"quelles clés le masque causal retire-t-il à la dernière requête ?": False})
wb.record("B3.2f", int(BOTTOM_RIGHT_B32[0].sum()), mistakes={"relis la question : on demande le masque de ta librairie, aligné en bas à droite": 1,
          "relis la docstring de causal_mask : la requête 0 est le token de quelle position ?": 3})
wb.record("B3.2g", int(TOP_LEFT_B32[0].sum()), mistakes={"relis la question : on demande l'alignement de is_causal=True quand L ≠ S": 2,
          "relis l'encadré 🕰️ sur F.scaled_dot_product_attention : où s'aligne le triangle ?": 3})
wb.record("B3.2h", False, mistakes={"écris la ligne 2 du masque transposé : quelles clés autorise-t-elle ?": True})

WB_ANSWER {"hash": "8acb045378c547c511749b8941024115c3267d0484dcff390f5c22fd10572eeb", "id": "B3.2a", "kind": "int", "magnitude": 0, "mistakes": {"34e38ced06107f4213a54c475143fa8d5f631e4746985de16faa99d36c5ed626": "le masque causal interdit des cases : lesquelles ?", "67f8954a8f233ba900aaf9a28fd3b5d9bb2fee2732cd2b551926803d2b151074": "la diagonale est-elle autorisée ?"}, "sign": 1}
📝 Ex B3.2a : réponse enregistrée (int).
WB_ANSWER {"decimals": 3, "element_coarse": [["6970041dc177d7f52cb880452d09e8eabf2e1a5fec378192281a39857bb023dd"], ["a37c54a22feda159f24ded31a0527550bb1c8dca5c9b6cdbfbfd9f899e38cef4"], ["523771d7d4bf75dd582869275392a3aa228412d1685636c9c6f954ff2f10ca6b"]], "element_hashes": ["ec362a3818d2208f9fb2ef714759f5b73f6bc9a95518bd59d537e3f91d443bc0", "c1bb86cb5b9d8efbb3f65ede5460d5821e333de225166211654975ef029e845c", "bee2a0bc1d96fe088cbb5e9787f813146b049cb3ad5f38aa14d4ea2cf0780b53"], "hash": "55d05e7e84fff6f010f9898142ab94236c7232150179ea2fbcb13aff470fc154", "id": "B3.2b", "kin

**Ex B3.3 — Suivre les formes dans l'attention multi-têtes et compter ses paramètres**

In [20]:
wb.record("B3.3a", "(4, 10, 384)", mistakes={"n'oublie pas l'axe des séquences du batch": "(10, 384)",
          "une seule matrice W_Q pour toutes les têtes : relis « Une seule matrice pour toutes les têtes »": "(4, 10, 64)",
          "relis la question : avant split_heads": "(4, 6, 10, 64)"})
wb.record("B3.3b", "(4, 6, 10, 64)", mistakes={"relis la docstring de split_heads : quel axe vient juste après N ?": "(4, 10, 6, 64)",
          "relis la docstring de split_heads : quel axe vient en premier ?": "(6, 4, 10, 64)",
          "relis : une tête a-t-elle toutes les features ?": "(4, 6, 10, 384)"})
wb.record("B3.3c", "(4, 6, 10, 10)", mistakes={"relis la docstring de multi_head_attention : une carte par tête ?": "(4, 10, 10)",
          "les poids ont une colonne par clé : relis l'ordre des axes de Q Kᵀ": "(4, 6, 10, 64)",
          "c'est la forme de la sortie : on demande les poids": "(4, 10, 384)"})
wb.record("B3.3d", N_B33 * H_B33 * T_B33 * T_B33, mistakes={"compte toutes les têtes": 400,
          "compte toutes les séquences du batch": 600,
          "relis : combien de scores pour une requête ?": 240})
wb.record("B3.3e", MHA_B33, mistakes={"n'oublie pas les quatre biais": 4 * D_B33 ** 2,
          "n'oublie pas la projection de sortie": 3 * D_B33 ** 2 + 3 * D_B33,
          "relis la forme de W_Q dans l'énoncé : (d, d)": 4 * D_B33 * DH_B33 + 4 * DH_B33})
wb.record("B3.3f", True, mistakes={"relis « Compter les paramètres » (§203.4)": False})
wb.record("B3.3g", "(1152, 384)", mistakes={"relis l'exemple de nn.MultiheadAttention(8, 2) (§203.4) : dans quel sens les matrices sont-elles rangées ?": "(384, 1152)",
          "relis : combien de matrices dans in_proj_weight ?": "(384, 384)",
          "relis : où est rangée W_O ?": "(1536, 384)"})
wb.record("B3.3h", GQA_B33, mistakes={"relis la question : les clés et les valeurs n'ont que 2 têtes": MHA_B33,
          "n'oublie pas les biais": GQA_B33 - 2 * D_B33 - 2 * HKV_B33 * DH_B33,
          "relis la question : W_Q et W_O sont comprises": 2 * (D_B33 * HKV_B33 * DH_B33 + HKV_B33 * DH_B33),
          "relis la forme de W_K et de W_V : (384, 128)": 2 * (D_B33 * D_B33 + D_B33) + 2 * (D_B33 * DH_B33 + DH_B33)})
wb.record("B3.3i", "(4, 6, 10, 25)", mistakes={"relis l'ordre des axes de Q Kᵀ : les lignes sont-elles les requêtes ou les clés ?": "(4, 6, 25, 10)",
          "relis la question : les clés viennent des 25 tokens de l'encodeur": "(4, 6, 10, 10)",
          "relis la docstring de multi_head_attention : une carte par tête ?": "(4, 10, 25)"})

WB_ANSWER {"hash": "be01e495be81c186e20934721bd880466bd30e633da08d8dad22b7bee4a0039f", "id": "B3.3a", "kind": "str", "mistakes": {"9e4abd968fb54e56976535ffab34b378214880f7e47126dd8cae93445d05738d": "une seule matrice W_Q pour toutes les têtes : relis « Une seule matrice pour toutes les têtes »", "a1952dad9b9bd4a72aa5d648d13261dd4f4bd77730f86ed74960552a4143f2a2": "n'oublie pas l'axe des séquences du batch", "b100a5f3ec6b5157e356f2c3b3bff0179a0834dda57567f8f2653cbc8d09a3c7": "relis la question : avant split_heads"}}
📝 Ex B3.3a : réponse enregistrée (str).
WB_ANSWER {"hash": "a18f754c306717c3ece366ff39a02c42494e4e137f2cf5e4afd6a860b13835be", "id": "B3.3b", "kind": "str", "mistakes": {"1e899e1e9748d707f859978a8a92a1e83bc75432e3f1dbaeec0398ae0da0a7c3": "relis la docstring de split_heads : quel axe vient en premier ?", "6a608a5bcce6b9837ad198312f185f40fe24085716368042beeb98c47c437acf": "relis : une tête a-t-elle toutes les features ?", "f8c1d934e39909856e191af389b80d4d8a3797afad9beebb1adbd96

**Ex B3.4 — Encodage sinusoïdal : quelques valeurs et l'effet d'un décalage de position**

In [21]:
wb.record("B3.4a", round(2 * math.pi / OMEGA_B34[2]), mistakes={"relis la période d'une horloge de fréquence angulaire ω (§203.5)": 100,
          "relis la question : la paire i = 2 (colonnes 4 et 5)": 63,
          "vérifie ω_2 : l'exposant 2i/d avec i = 2": 6283},
          fractional="arrondis la période à l'entier le plus proche")
wb.record("B3.4b", pe_b34(25)[0], decimals=3, mistakes={"les angles sont en radians": math.sin(math.radians(25)),
          "la colonne 0 est un sinus": math.cos(25),
          "vérifie le signe": -math.sin(25)})
wb.record("B3.4c", pe_b34(25)[2], decimals=4, mistakes={"vérifie la paire : la colonne 2 est le sinus de la paire i = 1": math.sin(0.25),
          "relis la question : les colonnes paires sont des sinus": math.cos(2.5),
          "les angles sont en radians": math.sin(math.radians(2.5))})
wb.record("B3.4d", pe_b34(25)[3], decimals=4, mistakes={"relis la question : les colonnes impaires sont des cosinus": math.sin(2.5),
          "vérifie la paire : la colonne 3 est le cosinus de la paire i = 1": math.cos(0.25),
          "vérifie le signe": -math.cos(2.5),
          "les angles sont en radians": math.cos(math.radians(2.5))})
wb.record("B3.4e", SHIFT_B34[0], decimals=4, mistakes={"relis la matrice du décalage de la fiche : vérifie le signe de chaque terme": shift_b34(PAIR_B34, -1.0)[0],
          "le décalage de la paire 1 se fait à sa propre fréquence, ω_1": shift_b34(PAIR_B34, 10.0)[0],
          "relis la question : on demande la colonne 2, le sinus": SHIFT_B34[1]})
wb.record("B3.4f", SHIFT_B34[1], decimals=4, mistakes={"relis la matrice du décalage de la fiche : vérifie le signe de chaque terme": shift_b34(PAIR_B34, -1.0)[1],
          "le décalage de la paire 1 se fait à sa propre fréquence, ω_1": shift_b34(PAIR_B34, 10.0)[1],
          "relis la question : on demande la colonne 3, le cosinus": SHIFT_B34[0]})
wb.record("B3.4g", PE_DOT_B34, decimals=4, mistakes={"relis la question : les deux positions sont différentes": 4.0,
          "les angles sont en radians": float(pe_b34(0) @ pe_b34(2, np.radians(OMEGA_B34))),
          "vérifie les fréquences : l'exposant est 2i/d": float(pe_b34(0) @ pe_b34(2, 10000.0 ** (-np.arange(4) / 8)))})
wb.record("B3.4h", True, mistakes={"relis la propriété du produit scalaire de deux encodages (§203.5)": False})

WB_ANSWER {"fractional": "arrondis la période à l'entier le plus proche", "hash": "4160a4d49faaea1c1b32ecbcb46800421eee53039081fbcee2d7bde2a8c8d08e", "id": "B3.4a", "kind": "int", "magnitude": 2, "mistakes": {"1e3c744a84283e6e28f87b8baeaf86ac9338be78811a17975e970a1ba74b1dbf": "vérifie ω_2 : l'exposant 2i/d avec i = 2", "74c5064c9453be934d7511e3ead6614a7f8ee6380ff1c846fb8e27d094dfcf9c": "relis la question : la paire i = 2 (colonnes 4 et 5)", "83df17e6f1785f41ec2046d8590b6987164da375ff1ed7caca83cd3c44ae4886": "relis la période d'une horloge de fréquence angulaire ω (§203.5)"}, "sign": 1}
📝 Ex B3.4a : réponse enregistrée (int).
WB_ANSWER {"decimals": 3, "hash": "18ab348e21ee46bb19d5a5ca36ad097cb070b41f792dab086afcfc5e74a98fe2", "hash_coarse": "3e7e0a6ac18785fdcc4f32122e0c6f445fbcff7d2979f2a1f5a19be1c9886745", "id": "B3.4b", "kind": "float", "magnitude": -1, "mistakes": {"53343077c0634694b5a89dfcaa89db824b0dfc2d1677ef9454b8782fde77fca4": "la colonne 0 est un sinus", "60f8082c28cefa96f2efdc

**Ex B3.5 — Compter les paramètres d'un GPT et retrouver les 124 millions de GPT-2 small**

In [22]:
wb.record("B3.5a", GPT2_B35["attn"], mistakes={"n'oublie pas les quatre biais": 4 * 768 ** 2,
          "n'oublie pas la projection de sortie": 3 * 768 ** 2 + 3 * 768})
wb.record("B3.5b", GPT2_B35["mlp"], mistakes={"n'oublie pas les biais des deux couches denses": 8 * 768 ** 2,
          "vérifie la taille du biais de la première couche": 8 * 768 ** 2 + 2 * 768,
          "vérifie la taille du biais de la seconde couche": 8 * 768 ** 2 + 2 * 3072,
          "relis la largeur du MLP : F = 4d": 2 * 768 ** 2 + 2 * 768})
wb.record("B3.5c", GPT2_B35["block"], mistakes={"n'oublie pas les deux layer norms": GPT2_B35["attn"] + GPT2_B35["mlp"],
          "relis le ch. 20 : combien de vecteurs de paramètres par layer norm ?": GPT2_B35["attn"] + GPT2_B35["mlp"] + 2 * 768,
          "on demande le compte exact, pas l'approximation 12d²": 12 * 768 ** 2})
wb.record("B3.5d", GPT2_B35["blocks"], mistakes={"on demande le compte exact, pas l'approximation 12d²": 12 * 12 * 768 ** 2,
          "relis la question : les 12 blocs seuls": GPT2_B35["non_embedding"]})
wb.record("B3.5e", GPT2_B35["wte"] + GPT2_B35["wpe"], mistakes={"relis la question : la table des positions aussi": GPT2_B35["wte"],
          "relis la question : la table des tokens aussi": GPT2_B35["wpe"],
          "la couche de sortie est liée : elle ne compte pas une seconde fois": 2 * GPT2_B35["wte"] + GPT2_B35["wpe"]})
wb.record("B3.5f", GPT2_B35["total"], mistakes={"relis l'énoncé : la couche de sortie réutilise la table des tokens": GPT2_UNTIED_B35,
          "n'oublie pas la layer norm finale": GPT2_B35["total"] - 2 * 768,
          "n'oublie pas la table des positions": GPT2_B35["total"] - GPT2_B35["wpe"],
          "c'est le nombre de l'article, que le dépôt dit faux : compte les paramètres": 117000000})
wb.record("B3.5g", GPT2_B35["non_embedding"], mistakes={"relis la question : la layer norm finale comprise": GPT2_B35["blocks"],
          "relis la question : sans la table des positions": GPT2_B35["non_embedding"] + GPT2_B35["wpe"],
          "relis la question : sans les tables d'embedding": GPT2_B35["total"]})
wb.record("B3.5h", 100 * (GPT2_B35["total"] - PAPER_B35) / PAPER_B35, decimals=1, mistakes={"rapporte l'écart aux 117 millions de l'article": 100 * (GPT2_B35["total"] - PAPER_B35) / GPT2_B35["total"],
          "on demande un pourcentage, pas une proportion": (GPT2_B35["total"] - PAPER_B35) / PAPER_B35,
          "c'est l'écart en millions : on demande un pourcentage": (GPT2_B35["total"] - PAPER_B35) / 1e6})
wb.record("B3.5i", GPT2_UNTIED_B35 > 160e6, mistakes={"relis ce que les poids liés économisent (B2, §202.7)": False})
wb.record("B3.5j", MINI_B35["total"], mistakes={"relis la question : on demande tous les paramètres, tables comprises": MINI_B35["non_embedding"],
          "relis l'énoncé : la couche de sortie est liée": gpt_params(4, 128, 88, 128, tied=False)["total"],
          "n'oublie pas la table des positions": MINI_B35["total"] - MINI_B35["wpe"],
          "relis la question : le compte exact, tables comprises": 4 * 12 * 128 ** 2})

WB_ANSWER {"hash": "46314762ad70ec6fd45e63396289ad269811f24c5ca3baab4d9339e17516398c", "id": "B3.5a", "kind": "int", "magnitude": 6, "mistakes": {"137736b1041967a4d81014dc4ef19329b10bd91dfa1f49f2b4d04518595df5b9": "n'oublie pas les quatre biais", "9e8df5f7076aa76b61af73b8bb9b2e32d35c5548cd46b2c3fecf11ffa963c3b9": "n'oublie pas la projection de sortie"}, "sign": 1}
📝 Ex B3.5a : réponse enregistrée (int).
WB_ANSWER {"hash": "0fa8e627082e595fc518fada5a88c2cc849fd7bf0624b2cc4e45c57de9db4ecb", "id": "B3.5b", "kind": "int", "magnitude": 6, "mistakes": {"33d90e546bef576dc79bc37e38debde00fe1f6725bb1fe3e2cec44f973094948": "relis la largeur du MLP : F = 4d", "97c30558501e90e3e5f7c7ea021464058b3f456841be2c5b47d664d6137bb363": "vérifie la taille du biais de la seconde couche", "b1bca395183ab357605208ffa40dc39b6253dc54051c76857c43187e587b7a7e": "n'oublie pas les biais des deux couches denses", "baa9289e821ad313117fc74ce73f9d7b291e3fa102602a02f7eb00d1017899bb": "vérifie la taille du biais de la prem

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une attention à la main, masque causal compris, et dire pourquoi on divise par $\sqrt{d_k}$ ?
2. Sais-tu suivre les formes d'une attention multi-têtes et compter les paramètres d'un GPT ?
3. Sais-tu dire ce que font un encodage sinusoïdal et RoPE, et démontrer la propriété de RoPE ?
4. Sais-tu chiffrer le calcul d'un entraînement et la mémoire d'un KV cache ?

**Pour aller plus loin** : la vidéo « Let's build GPT » d'A. Karpathy et *The Annotated Transformer*, cités dans la fiche. La suite : le bonus B4, les LLM en pratique.

*Les exercices de code B3.12 à B3.30 (partie A : ta librairie NumPy ; partie B : le mini-GPT en PyTorch) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B3` pour obtenir le notebook complet.*